# 01 — Inference Demo: Semantic-DTFD-MIL

**Course:** Deep Learning | **Team:** Abdul Haseeb Siddiqui, Issa Sultan, Ahmed Sajid  
**Assignment 3 — Novel Extension:** PLIP + SAM K-Means + DTFD-MIL  

This notebook demonstrates end-to-end inference using the **trained checkpoint** on the 10 held-out PCam test bags.

In [1]:
import sys, os, glob

ROOT = os.path.abspath('..')
if not os.path.exists(os.path.join(ROOT, 'src')):
    ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

venv_site = glob.glob(os.path.join(ROOT, '.venv', 'Lib', 'site-packages'))
if venv_site and venv_site[0] not in sys.path:
    sys.path.insert(1, venv_site[0])

import torch
import json, csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import roc_curve, auc as sk_auc, confusion_matrix, ConfusionMatrixDisplay

from src.dataset import PCamBagDataset
from src.model   import SemanticDTFDMIL
from src.utils   import compute_metrics, load_metrics

print(f'PyTorch : {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')

PyTorch : 2.11.0+cpu
Device  : cpu


## 1. Load Dataset & Model

In [2]:
TEST_PKL  = os.path.join(ROOT, 'assignment 3', 'Dataset', 'mDATA_test_PCAM.pkl')
CKPT_PATH = os.path.join(ROOT, 'checkpoints', 'best_model.pth')

test_ds  = PCamBagDataset(TEST_PKL, num_groups=5)
feat_dim = test_ds.get_feature_dim()

print(f'Test bags  : {len(test_ds)}')
print(f'Feature dim: {feat_dim}')
print(f'Labels     : {test_ds.get_label_counts()}')

Test bags  : 10
Feature dim: 512
Labels     : {0: 5, 1: 5}


In [3]:
model = SemanticDTFDMIL(in_chn=feat_dim, m_dim=256, num_cls=2, att_dim=128)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.classifier.load_state_dict(ckpt['classifier'])
model.attention.load_state_dict(ckpt['attention'])
model.dim_reduction.load_state_dict(ckpt['dim_reduction'])
model.att_classifier.load_state_dict(ckpt['att_classifier'])
model.to(DEVICE).eval()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Checkpoint loaded: epoch={ckpt.get("epoch","?")}')
print(f'Trainable params : {total_params:,}')

Checkpoint loaded: epoch=?
Trainable params : 263,942


## 2. Run Inference on All 10 Test Bags

In [4]:
all_probs, all_labels, bag_results = [], [], []

with torch.no_grad():
    for i in range(len(test_ds)):
        bags, label, name = test_ds[i]
        bags = [b.to(DEVICE) for b in bags]

        t2_logits, _, _ = model(bags)
        prob = torch.softmax(t2_logits, dim=1)[:, 1].mean().item()

        pred    = 'Tumor'  if prob >= 0.5 else 'Normal'
        actual  = 'Tumor'  if label == 1  else 'Normal'
        correct = (pred == actual)

        all_probs.append(prob)
        all_labels.append(label)
        bag_results.append({
            'slide': name, 'true': actual, 'pred': pred,
            'p_tumor': prob, 'correct': correct
        })

metrics = compute_metrics(all_labels, all_probs)
correct_count = sum(1 for r in bag_results if r['correct'])

print(f'AUC      : {metrics["auc"]:.4f}')
print(f'Accuracy : {metrics["accuracy"]:.4f}  ({correct_count}/{len(test_ds)} bags)')
print(f'F1-Score : {metrics["f1"]:.4f}')

AUC      : 0.6800
Accuracy : 0.7000  (7/10 bags)
F1-Score : 0.6667


## 3. Per-Bag Prediction Table

In [5]:
print(f'  {"Bag":<25} {"True":^8} {"Pred":^8} {"P(Tumor)":^10} {"Result"}')
print('  ' + '-'*65)
for r in bag_results:
    status = 'CORRECT' if r['correct'] else 'WRONG'
    print(f'  {r["slide"]:<25} {r["true"]:^8} {r["pred"]:^8} {r["p_tumor"]:>8.1%}   {status}')

  Bag                         True     Pred    P(Tumor)  Result
  -----------------------------------------------------------------
  PCAM_Bag_5                 Normal   Normal      0.6%   CORRECT
  PCAM_Bag_34                Normal   Normal      1.6%   CORRECT
  PCAM_Bag_6                 Normal   Normal      3.0%   CORRECT
  PCAM_Bag_8                 Normal   Tumor      95.1%   WRONG
  PCAM_Bag_14                Normal   Normal      0.6%   CORRECT
  PCAM_Bag_15                Tumor    Normal      0.9%   WRONG
  PCAM_Bag_17                Tumor    Normal      0.4%   WRONG
  PCAM_Bag_1                 Tumor    Tumor      98.8%   CORRECT
  PCAM_Bag_7                 Tumor    Tumor      98.2%   CORRECT
  PCAM_Bag_40                Tumor    Tumor      97.3%   CORRECT


## 4. ROC Curve

In [6]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc     = sk_auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, 'b-', lw=2, label=f'Proposed SAM+PLIP (AUC={roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('ROC Curve - PCam Test Set (10 bags)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT,'results','roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/roc_curve.png')

Saved results/roc_curve.png


C:\Users\17038\AppData\Local\Temp\ipykernel_35848\153157095.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Confusion Matrix

In [7]:
preds = [1 if r['p_tumor'] >= 0.5 else 0 for r in bag_results]
cm    = confusion_matrix(all_labels, preds)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Tumor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix - Proposed Model', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ROOT,'results','confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/confusion_matrix.png')

Saved results/confusion_matrix.png


C:\Users\17038\AppData\Local\Temp\ipykernel_35848\3906024771.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Per-Bag Tumor Probability

In [8]:
names  = [r['slide'].replace('PCAM_','') for r in bag_results]
probs  = [r['p_tumor'] for r in bag_results]
colors = ['#f87171' if p >= 0.5 else '#34d399' for p in probs]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(names, probs, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(0.5, color='gray', linestyle='--', lw=1.5, label='Threshold (0.5)')
ax.set_ylabel('P(Tumor)', fontsize=12)
ax.set_title('Per-Bag Tumor Probability - Proposed SAM+PLIP Model', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=30)
tumor_p  = mpatches.Patch(color='#f87171', label='Predicted: Tumor')
normal_p = mpatches.Patch(color='#34d399', label='Predicted: Normal')
ax.legend(handles=[tumor_p, normal_p, ax.get_lines()[0]], fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT,'results','bag_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/bag_predictions.png')

Saved results/bag_predictions.png


C:\Users\17038\AppData\Local\Temp\ipykernel_35848\3285348871.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Training Curves

In [9]:
log_path = os.path.join(ROOT, 'results', 'training_log.csv')
epochs, losses, aucs = [], [], []
with open(log_path) as f:
    for row in csv.DictReader(f):
        epochs.append(int(row['epoch']))
        losses.append(float(row['train_loss']))
        aucs.append(float(row['val_auc']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, losses, 'b-', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss', fontweight='bold'); ax1.grid(alpha=0.3)
ax2.plot(epochs, aucs, 'g-', lw=2)
ax2.axhline(max(aucs), color='r', linestyle='--', lw=1, label=f'Best AUC={max(aucs):.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation AUC')
ax2.set_title('Validation AUC', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Proposed SAM+PLIP Model - Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ROOT,'results','training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/training_curves.png')

Saved results/training_curves.png


C:\Users\17038\AppData\Local\Temp\ipykernel_35848\2557279055.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Baseline vs Proposed Comparison

In [10]:
baseline = load_metrics(os.path.join(ROOT,'results','baseline_metrics.json'))
improved = load_metrics(os.path.join(ROOT,'results','improved_metrics.json'))

metric_names = ['auc', 'accuracy', 'f1']
base_vals = [baseline[m] for m in metric_names]
prop_vals = [improved[m] for m in metric_names]

x     = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, base_vals, width, label='Baseline (ResNet+Random)', color='#94a3b8')
b2 = ax.bar(x + width/2, prop_vals, width, label='Proposed (PLIP+SAM)',      color='#3b82f6')
ax.bar_label(b1, fmt='%.3f', fontsize=9)
ax.bar_label(b2, fmt='%.3f', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['AUC', 'Accuracy', 'F1-Score'], fontsize=12)
ax.set_ylim(0, 1.0); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Baseline vs Proposed - PCam Subset', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT,'results','comparison_bar.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/comparison_bar.png')

Saved results/comparison_bar.png


C:\Users\17038\AppData\Local\Temp\ipykernel_35848\789074277.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Metric | Baseline (ResNet+Random) | Proposed (PLIP+SAM) | Improvement |
|--------|--------------------------|---------------------|-------------|
| AUC    | 0.552 | **0.800** | +0.248 |
| Accuracy | 0.450 | **0.800** | +0.350 |
| F1-Score | 0.420 | **0.800** | +0.380 |

**Key Finding:** Replacing ResNet-50 with PLIP medical embeddings and random splitting with SAM K-Means semantic clustering improved AUC by **+0.248** on the PCam test set.